In [1]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.13.0+cpu
0.28.0+cpu


In [2]:
pip install langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
from pathlib import Path

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama

c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Administrator\AppData\Local\Temp\ipykernel_15260\4192973880.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
PROJECT_ROOT = Path.cwd().parent

VECTOR_STORE = PROJECT_ROOT / "vector_store"

In [5]:
import torch
print(torch.__version__)

2.13.0+cpu


In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

In [7]:
cv_vectorstore = FAISS.load_local(
    VECTOR_STORE / "cv_index",
    embedding_model,
    allow_dangerous_deserialization=True
)


jd_vectorstore = FAISS.load_local(
    VECTOR_STORE / "jd_index",
    embedding_model,
    allow_dangerous_deserialization=True
)


In [8]:
cv_retriever = cv_vectorstore.as_retriever(
    search_kwargs={"k":3}
)

jd_retriever = jd_vectorstore.as_retriever(
    search_kwargs={"k":3}
)

In [9]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [10]:
query = "candidate skills experience education projects"

cv_docs = cv_retriever.invoke(query)

cv_context = "\n\n".join(
    doc.page_content for doc in cv_docs
)

print(cv_context[:2000])

& Stanford University via Coursera), with hands-on experience in regression, classification, and performance evaluation. EDUCATION King Fahd Languages Schools 07/2023 Level 3 Sadat Academy for Management Sciences, Faculty of Computer and Information With grade (Very Good) GPA (3.3). SKILLS Programming Python, C++, HTML, CSS Tools Microsoft Office (Excel, PowerPoint, Word) Technologies Machine Learning, Deep Learning Soft skills •Strong communication skills •Time management & organization •Problem-solving •Teamwork & collaboration •Adaptability & fast learning •Critical thinking •Attention to detail •Ability to work under pressure •Leadership & negotiation skills LANGUAGES Arabic Native German Beginner English Very Good CERTIFICATES ICT Hub Egypt — Machine Learning & AI Certificate of

& negotiation skills LANGUAGES Arabic Native German Beginner English Very Good CERTIFICATES ICT Hub Egypt — Machine Learning & AI Certificate of Achievement Aug 2025 CIB Summer Bootcamp — Digital Transfor

In [11]:
query = "required skills responsibilities qualifications"

jd_docs = jd_retriever.invoke(query)

jd_context = "\n\n".join(
    doc.page_content for doc in jd_docs
)

print(jd_context[:2000])

AI projects and results. Required Technical Skills • Python • Machine Learning • Deep Learning • TensorFlow • PyTorch • Scikit-learn • NumPy • Pandas • Data Preprocessing • Model Evaluation • Git • SQL Preferred Skills • LangChain • Retrieval-Augmented Generation (RAG) • Hugging Face Transformers • Docker • FastAPI • Streamlit • Computer Vision • Natural Language Processing (NLP) • Linux Soft Skills • Problem Solving • Critical Thinking • Communication • Teamwork • Time Management • Continuous Learning Education Bachelor's degree in Computer Science, Artificial Intelligence, Data Science, or a related field. Experience • 0–2 years of experience in AI or Machine Learning. • Internship experience is a plus. • Personal AI projects are highly valued. Tools & Technologies • Python • VS Code •

AI Engineer Job Description Job Title Artificial Intelligence Engineer Job Summary We are looking for an AI Engineer to develop intelligent applications using Machine Learning and Deep Learning techni

In [12]:
prompt = f"""

You are an AI HR assistant.

Analyze the candidate CV compared to the Job Description.

Candidate CV:
----------------
{cv_context}

Job Description:
----------------
{jd_context}


Return ONLY valid JSON with this format:

{{
    "match_score": 0,
    "matching_skills": [],
    "missing_skills": [],
    "strengths": [],
    "weaknesses": [],
    "recommendations": [],
    "final_decision": ""
}}

Rules:
- match_score must be between 0 and 100.
- Be realistic.
- Do not add any text outside JSON.

"""

In [13]:
response = llm.invoke(prompt)

result = response.content

print(result)

 {
    "match_score": 75,
    "matching_skills": ["Python", "Machine Learning", "Deep Learning", "Data Preprocessing", "Model Evaluation", "Git"],
    "missing_skills": ["TensorFlow", "Pytorch", "Scikit-learn", "NumPy", "Pandas", "Jupyter Notebook", "VS Code", "Google Colab", "Hugging Face", "FAISS", "LangChain"],
    "strengths": ["Strong problem-solving skills", "Teamwork and collaboration", "Communication skills", "Adaptability & fast learning", "Critical thinking", "Attention to detail", "Ability to work under pressure", "Leadership & negotiation skills"],
    "weaknesses": ["Lack of experience with some required technical skills (TensorFlow, Pytorch, Scikit-learn, NumPy, Pandas)", "No experience with LLMs or Prompt Engineering", "No cloud platforms experience (AWS, Azure, GCP)"],
    "recommendations": ["Consider gaining more hands-on experience with TensorFlow, PyTorch, Scikit-learn, NumPy, and Pandas to strengthen your technical skills.", "Explore learning about LLMs and Prompt 

In [14]:
try:
    result_json = json.loads(result)

    print(
        json.dumps(
            result_json,
            indent=4,
            ensure_ascii=False
        )
    )

except Exception as e:
    print("JSON Error:", e)

{
    "match_score": 75,
    "matching_skills": [
        "Python",
        "Machine Learning",
        "Deep Learning",
        "Data Preprocessing",
        "Model Evaluation",
        "Git"
    ],
    "missing_skills": [
        "TensorFlow",
        "Pytorch",
        "Scikit-learn",
        "NumPy",
        "Pandas",
        "Jupyter Notebook",
        "VS Code",
        "Google Colab",
        "Hugging Face",
        "FAISS",
        "LangChain"
    ],
    "strengths": [
        "Strong problem-solving skills",
        "Teamwork and collaboration",
        "Communication skills",
        "Adaptability & fast learning",
        "Critical thinking",
        "Attention to detail",
        "Ability to work under pressure",
        "Leadership & negotiation skills"
    ],
    "weaknesses": [
        "Lack of experience with some required technical skills (TensorFlow, Pytorch, Scikit-learn, NumPy, Pandas)",
        "No experience with LLMs or Prompt Engineering",
        "No cloud plat

In [15]:
with open(
    PROJECT_ROOT / "matching_result.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result_json,
        f,
        indent=4,
        ensure_ascii=False
    )